# Prompt Anatomy and Secure Model Connectivity

### What is an LLM


![](./images/llm_architecture_basic.png)

**Basic LLM Architecture**


#### LLM Model stats:

- What are parameters of an LLM? 8B, 100B etc..

- (Tokens per Second) TPS means? 

#### **Prompt:**

- Natural language and context payload supplied to LLM to steer its auto generative token generation

### Core Anatomy of an Enterprise Prompt

1. Instruction: The primary imperative task directive telling the model what action to execute (e.g., "Extract entities", "Summarize text").  

2. Context / Persona: Background information, operational boundaries, or behavioral framing (e.g., "You are an enterprise data engineer").  

3. Input Data: The variable payload or raw source text to be processed, typically isolated within explicit delimiters (""" or ###).  

4. Output Indicator: Formatting constraints and structural targets specifying the exact response schema (e.g., valid JSON schema, Markdown table, or bullet points).

### The Master Prompt Template (Open AI)
leverages structured formatting like XML tags to separate the role, context, task, and formatting rules

### SYSTEM PERSONA
You are an enterprise AI data ingestion engine specializing in customer support automation. 
Your job is to extract triage metadata from incoming customer tickets with high deterministic precision. 
Adhere strictly to the provided output schema and never invent or infer data not grounded in the source text.

### INSTRUCTION
1. Analyze the customer support ticket provided in the payload.
2. Identify and extract:
   - Primary customer issue
   - Urgency level (`LOW`, `MEDIUM`, `HIGH`, `CRITICAL`)
   - Sentiment score (`POSITIVE`, `NEUTRAL`, `NEGATIVE`)
   - Mentioned products/services
   - Action items for the support team
3. If information for a field is missing, set its value to `null`.
4. Suppress all conversational preamble, explanations, and markdown commentary outside the requested schema.

### INPUT DATA
"""
Ticket ID: #INC-94821
User: devops-lead@enterprise-client.com
Timestamp: 2026-08-23T10:14:00Z
Message:
Our production cluster on AWS us-east-1 went down 15 minutes ago after we pushed the latest database migration. The pgvector extension is throwing memory allocation errors, causing all semantic search queries to fail with HTTP 500 status codes. This is blocking our core checkout service. We need immediate assistance from the database infrastructure team to rollback or resize the instance.
"""

### OUTPUT INDICATOR
Respond strictly with valid JSON conforming to the following structure:
{
  "ticket_id": "string",
  "urgency": "LOW" | "MEDIUM" | "HIGH" | "CRITICAL",
  "sentiment": "POSITIVE" | "NEUTRAL" | "NEGATIVE",
  "issue_summary": "string",
  "affected_components": ["string"],
  "action_items": ["string"]
}

### Google GenAI Python SDK

In [2]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("Gemini API KEY not found!")

In [3]:
client = genai.Client(api_key=gemini_api_key)
prompt = "Tell me a joke on AI Engineering"

response = client.models.generate_content(
    model = "gemini-3.7-flash",
    contents = prompt
)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [4]:
print(response.text)

How can you tell the difference between a Junior AI Engineer and a Senior AI Engineer?

**Junior AI Engineer:**  
*"Extract the data from this document and output only valid JSON."*

**Senior AI Engineer:**  
*"Extract the data from this document and output only valid JSON. Take a deep breath, think step-by-step, I will tip you $200, and my grandmother's life depends on you not adding markdown backticks."*


### Primary API call Methods

- Standard Invocation: client.models.generate_content(model="gemini-2.5-flash", contents=...)  

- Streaming Protocol: client.models.generate_content_stream(model="gemini-2.5-flash", contents=...) 

- Async Invocation (FastAPI Integration): await client.aio.models.generate_content(...)  

- Vector Embeddings (RAG / Session 20): client.models.embed_content(model="text-embedding-004", contents=...) 

** Model Configurations: Passed via google.genai.types.GenerateContentConfig (handles system_instruction, temperature, max_output_tokens, and response_schema)

1. Standard Synchronous Call (Blocking)

Architecture: Blocks the executing Python thread until the model generates the entire response.  

Mechanism: The client sends an HTTP POST request to the inference endpoint and deserializes the full JSON payload upon completion.

When to Use: Batch scripts, data transformation jobs, or headless backend tasks where real-time user feedback is unnecessary.

In [ ]:
#standard Synchronous API

2. Streaming Call (generate_content_stream)

Architecture: Establishes a persistent Server-Sent Events (SSE) connection.  

Mechanism: Instead of waiting for 500+ tokens to compute, the server yields chunks as soon as attention decode layers finish sampling them. This drastically cuts Time-to-First-Token (TTFT) from ~1.5s down to ~150ms.  

When to Use: Interactive user interfaces/ Dashborads and real-time terminal outputs.

In [ ]:
#Streaming API
from google import genai
client = genai.Client()

response_stream = client.models.generate_content_stream(
    model = "gemini-3.7-flash",
    contents="Explain what is Model Context Proptocol (MCP) in detail"
)

#print(response_stream.text)
for chunk in response_stream:
    if chunk.text:
        print(chunk.text, end="", flush=True)



The **Model Context Protocol (MCP)** is an open-source standard introduced by Anthropic in late 2024. It is designed to solve one of the biggest challenges in AI development: **how to securely and seamlessly connect AI models (LLMs) to external data sources, local files, and business tools.**

To understand MCP easily, think of it as the **"USB-C port for AI applications."** 

Just as USB-C replaced dozens of proprietary cables with a single standard to connect any peripheral to any computer, MCP allows any AI model to connect to any data source or tool using a single, unified protocol.

---

### 1. The Problem MCP Solves

Before MCP, connecting an AI model to outside data required custom, fragmented integrations:
* If you wanted an AI to read a **PostgreSQL database**, **GitHub repository**, and **Slack workspace**, developers had to write three entirely different API wrappers.
* If a new, better AI model came out, developers often had to rewrite or adapt those connections.
* This cre

3. Asynchronous Invocation (client.aio)

Architecture: Integrates with Python’s asyncio event loop without blocking the main worker thread.  

Mechanism: Yields execution during the network I/O wait, allowing a single server worker to handle hundreds of concurrent requests simultaneously.  

When to Use: FastAPI Microservices, cyclic state machines (LangGraph), and concurrent web scrapers.

In [ ]:
#Asynch API

4. Structured Schema Enforcement (response_schema & Pydantic)

Architecture: Constrains the sampling logits at the decode layer to strictly conform to a JSON grammar/schema.

Mechanism: Guarantees that the output parses directly into a Pydantic object without requiring brittle regex or JSON-repair fallbacks.  

When to Use: Data extraction pipelines, database ingestion, and agent tool parameter extraction. 

In [8]:
#Structured Schema Enforcement using Pydantic
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

client = genai.Client()

class Incident(BaseModel):
    urgency: str = Field(description="Low, Medium, High")
    affected_service: str
    action_items: list[str]

response = client.models.generate_content(
    model = "gemini-3.7-flash",
    contents = "PostgreSQL is out of connections",
    config = types.GenerateContentConfig(
        response_mime_type = "application/json",
        response_schema=Incident,
        temperature = 0.0
    )
)

In [9]:
print(response.text)

{
  "urgency": "High",
  "affected_service": "PostgreSQL",
  "action_items": [
    "Identify and terminate idle or hung connections via pg_stat_activity",
    "Temporarily increase the max_connections setting in postgresql.conf and reload configuration",
    "Deploy or verify connection pooler (e.g., PgBouncer) settings",
    "Investigate application services for database connection leaks"
  ]
}


5. Dense Vector Embeddings (embed_content)

Architecture: Queries the text representation layer to produce fixed-dimension dense vectors (D=768).  

Mechanism: Transforms raw semantic text into vector space coordinates used for nearest-neighbor search (Cosine/Dot Product) in Phase 4 (RAG).  

When to Use: Ingesting documents into ChromaDB, Pinecone, or PostgreSQL (pgvector) (Sessions 19–21).

In [10]:
#vector embedding API
result = client.models.embed_content(
    model = "gemini-embedding-001",
    contents= "The number of keys in a keyboard can vary from 105 to 120"   
)

In [15]:
vector = result.embeddings[0].values
print(len(vector))

3072


6. Interactions API (Recommended/latest)

- Google’s unified interface across the google-genai SDK for interacting with both foundation models and autonomous agents (such as Deep Research).
- Interactions API introduces server-side state management.
- Conversation Forking: Branch a discussion into multiple paths from any historical interaction.id without state mutation.
- Native Multimodal Generation: Supports text, image, and audio outputs within the same execution flow.

## Case Study: Stateful LLM Chat

**Scenario:** You got a SMS stating your credit card was debited with Rs. 10,000 without your knowledge. Get help from gemini falsh LLM to Investigate the transaction, without sending same prompt texts to and fro.

In [16]:
#Intercations API

import os
from dotenv import load_dotenv
from google import genai

# 1. Load API Key and initialize client
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 2. Turn 1: Initial user symptom report
print("--- [Turn 1: User Reports Issue] ---")
turn_1 = client.interactions.create(
    model="gemini-2.5-flash",
    input="Hi, I got an SMS stating that Rs.10,000 was debited from my credit card without my consent."
)



--- [Turn 1: User Reports Issue] ---


In [ ]:
# Extract generated response text
print(f"Assistant: {turn_1.outputs[-1].text}\n")
print(f"Interaction ID: {turn_1.id}")

# 3. Turn 2: Follow-up question referencing only previous_interaction_id
print("\n--- [Turn 2: Asking for Remediation without re-sending history] ---")
turn_2 = client.interactions.create(
    model="gemini-2.5-flash",
    input="What should I do to make sure my credit card will not be misused?",
    previous_interaction_id=turn_1.id  # Server automatically injects Turn 1 context
)

print(f"Assistant: {turn_2.outputs[-1].text}\n")

# 4. Turn 3 (Forking/Branching): Asking an alternative question from Turn 1
print("\n--- [Turn 3: Branching/Forking Context from Turn 1] ---")
turn_3_fork = client.interactions.create(
    model="gemini-2.5-flash",
    input="May be my wife used it for shopping, how to know without hurting her feelings",
    previous_interaction_id=turn_1.id  # Directly branches from Turn 1
)

print(f"Assistant (Forked Branch): {turn_3_fork.outputs[-1].text}")